# CHALLENGE 01 — Diseña tu chunking

**Tiempo sugerido:** 20 minutos · **No requiere API key** · Trabajo en parejas.

Chunking no es "corta cada X caracteres": es una **decisión de arquitectura**.
Tu reto es analizar el documento y **justificar** una estrategia antes de escribir el código (eso será el LAB 02).

Debes decidir:
- ¿Qué tamaño de chunk utilizarías?
- ¿Qué overlap utilizarías?
- ¿Dividirías por caracteres, palabras, párrafos o secciones?
- ¿Qué problemas podría producir tu estrategia?

⛔ No abras la solución hasta discutir tu propuesta.

In [ ]:
# Verificación del entorno: este notebook debe correr con el kernel .venv del repositorio
import sys, importlib.util
_faltan = [p for p in ["numpy", "pandas", "sklearn", "tiktoken", "openai", "dotenv", "matplotlib"]
           if importlib.util.find_spec(p) is None]
if _faltan:
    raise ModuleNotFoundError(
        f"Faltan paquetes: {', '.join(_faltan)}\n"
        f"Este kernel usa: {sys.executable}\n"
        "Solución: en la terminal, dentro del repositorio, ejecuta `uv sync`; luego en VS Code elige "
        "Select Kernel → Python Environments → .venv (la del repositorio) y reinicia el kernel."
    )
print("✅ Entorno listo:", sys.executable)

In [ ]:
# Configuración común: localiza el repositorio, carga .env y el documento de trabajo
import os
import json
from pathlib import Path
from dotenv import load_dotenv

def encontrar_raiz():
    p = Path.cwd().resolve()
    for c in [p, *p.parents]:
        if (c / "data").is_dir() and (c / "pyproject.toml").exists():
            return c
    raise FileNotFoundError("No encontré la carpeta data/. Abre el notebook desde el repositorio del curso.")

ROOT = encontrar_raiz()
load_dotenv(ROOT / ".env")
DATA_DIR = ROOT / "data"

DOCUMENTOS = {
    "cafeteria":   ("cafeteria_nebula_manual.md",      "cafeteria_preguntas.json"),
    "streamly":    ("streamly_api_docs.md",            "streamly_preguntas.json"),
    "veterinaria": ("clinica_veterinaria_huellitas.md", "veterinaria_preguntas.json"),
    "mi_documento": ("mi_documento.md",                 None),  # genera el tuyo con los prompts de alumnos/02_material_del_alumno.md
}

def cargar_documento(nombre):
    archivo, _ = DOCUMENTOS[nombre]
    return (DATA_DIR / archivo).read_text(encoding="utf-8")

def cargar_preguntas(nombre):
    _, archivo = DOCUMENTOS[nombre]
    if archivo is None:
        return []
    return json.loads((DATA_DIR / archivo).read_text(encoding="utf-8"))

print("Repositorio:", ROOT)

In [ ]:
# 👉 Elige el documento con el que vas a trabajar
DOCUMENTO = "cafeteria"   # "cafeteria" | "streamly" | "veterinaria" | "mi_documento"

texto = cargar_documento(DOCUMENTO)
print(f"Documento: {DOCUMENTO} — {len(texto):,} caracteres, {len(texto.split()):,} palabras")
print(texto[:500], "...")

## 1. Radiografía del documento

In [ ]:
import re
import pandas as pd
import tiktoken

enc = tiktoken.get_encoding("o200k_base")
lineas = texto.splitlines()
secciones = [l for l in lineas if re.match(r"^#{1,6}\s", l)]
parrafos = [p for p in texto.split("\n\n") if p.strip()]
tablas = sum(1 for l in lineas if l.strip().startswith("|"))
bloques_codigo = texto.count("```") // 2

print(f"Caracteres: {len(texto):,} | Tokens: {len(enc.encode(texto)):,}")
print(f"Secciones (encabezados): {len(secciones)} | Párrafos: {len(parrafos)}")
print(f"Líneas de tabla: {tablas} | Bloques de código: {bloques_codigo}")
print("\nEncabezados:")
for s in secciones:
    print("  ", s)

largos = pd.Series([len(p) for p in parrafos], name="caracteres por párrafo")
largos.describe().round(0)

## 2. Herramienta de vista previa

Esta función ya está hecha: te deja **probar** una estrategia sin programar todavía.

In [ ]:
def vista_previa(estrategia="caracteres", chunk_size=500, overlap=50, mostrar=3):
    if estrategia == "caracteres":
        paso = chunk_size - overlap
        chunks = [texto[i:i + chunk_size] for i in range(0, len(texto), paso)]
    elif estrategia == "palabras":
        pal = texto.split()
        paso = chunk_size - overlap
        chunks = [" ".join(pal[i:i + chunk_size]) for i in range(0, len(pal), paso)]
    elif estrategia == "parrafos":
        chunks = parrafos
    elif estrategia == "secciones":
        chunks = [c for c in re.split(r"\n(?=#{1,6}\s)", texto) if c.strip()]
    else:
        raise ValueError("estrategia: caracteres | palabras | parrafos | secciones")
    tam = [len(c) for c in chunks]
    print(f"{estrategia}: {len(chunks)} chunks | min {min(tam)} | prom {sum(tam)//len(tam)} | max {max(tam)} caracteres")
    for c in chunks[:mostrar]:
        print("─" * 80)
        print(c[:600])
    return chunks

_ = vista_previa("caracteres", 300, 0)

In [ ]:
# Prueba otras estrategias
_ = vista_previa("secciones", mostrar=2)

### Busca los puntos frágiles

¿Qué le pasa a una **tabla** o a un **bloque de código** con tu estrategia? Encuentra un chunk que contenga `|` y revísalo.

In [ ]:
chunks = vista_previa("caracteres", 300, 0, mostrar=0)
con_tabla = [c for c in chunks if c.count("|") > 4]
print(f"{len(con_tabla)} chunks contienen fragmentos de tablas. Ejemplo:\n")
print(con_tabla[0] if con_tabla else "(sin tablas)")

## 3. Tu ficha de diseño

In [ ]:
ficha = {
    "documento": DOCUMENTO,
    "unidad": "",          # TODO: caracteres | palabras | tokens | parrafos | secciones (o combinación)
    "chunk_size": None,    # TODO
    "overlap": None,       # TODO
    "justificacion": "",   # TODO: ¿por qué? piensa en las preguntas que hará el usuario
    "riesgos": [],         # TODO: ¿qué podría salir mal? (tablas, listas, secciones muy largas...)
    "como_lo_probarias": "",
}
ficha

## Discusión (5 min)

1. Compartan su ficha con otra pareja. ¿Eligieron lo mismo? ¿Por qué no?
2. ¿Su estrategia funcionaría igual para un documento de 500 páginas? ¿Y para un chat de WhatsApp exportado?
3. Si el documento tiene tablas de datos o listas numéricas largas, ¿qué problema aparece?